# Text-Prompted Segmentation with LangSAM

[LangSAM](https://github.com/luca-medeiros/lang-segment-anything) combines Grounding DINO (an open-vocabulary object detector) with SAM. You supply a **text prompt** such as `"building"`, `"tree"`, or `"road"`, Grounding DINO finds all matching objects in the image, and SAM generates precise segmentation masks for each detected box.

samgeo wraps LangSAM with full geospatial support: input is a GeoTIFF and output masks carry the original CRS and geotransform.

## References
- samgeo LangSAM tutorial: https://samgeo.gishub.org/examples/text_prompts/
- Grounding DINO paper: https://arxiv.org/abs/2303.05499

## 1. Setup

In [ ]:
import os
import urllib.request
import matplotlib.pyplot as plt
from samgeo.text_sam import LangSAM

url = "https://github.com/opengeos/samgeo/releases/download/v0.2.0/aerial_image.tif"
image_path = "aerial_image.tif"
if not os.path.exists(image_path):
    urllib.request.urlretrieve(url, image_path)

# LangSAM downloads Grounding DINO weights on first use (~700 MB)
sam = LangSAM()

## 2. Segment with a Text Prompt

In [ ]:
text_prompt = "building"

sam.predict(
    image=image_path,
    text_prompt=text_prompt,
    box_threshold=0.24,   # Grounding DINO detection confidence threshold
    text_threshold=0.24,
    output="buildings_mask.tif",
)
print(f"Segmented '{text_prompt}' instances saved to buildings_mask.tif")

## 3. Visualize Results

In [ ]:
sam.show_anns(
    cmap="Greens",
    add_boxes=True,
    alpha=0.5,
    title=f"LangSAM: '{text_prompt}'",
    blend=True,
    output="buildings_viz.png",
)

img = plt.imread("buildings_viz.png")
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Compare Different Prompts

In [ ]:
prompts = ["tree", "road", "car"]
outputs = []

for prompt in prompts:
    out_path = f"{prompt}_mask.tif"
    sam.predict(
        image=image_path,
        text_prompt=prompt,
        box_threshold=0.24,
        text_threshold=0.24,
        output=out_path,
    )
    viz_path = f"{prompt}_viz.png"
    sam.show_anns(cmap="Reds", alpha=0.5, title=f"'{prompt}'", blend=True, output=viz_path)
    outputs.append(viz_path)

fig, axes = plt.subplots(1, len(prompts), figsize=(6 * len(prompts), 6))
for ax, viz_path, prompt in zip(axes, outputs, prompts):
    ax.imshow(plt.imread(viz_path))
    ax.set_title(f"'{prompt}'")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Export to GeoJSON

In [ ]:
from samgeo import SamGeo

# Reuse SamGeo's vector export utility
sg = SamGeo(model_type="vit_b", automatic=False)
sg.tiff_to_vector("buildings_mask.tif", "buildings.geojson")
print("Exported building footprints to buildings.geojson")

## Tuning Tips

- **`box_threshold`**: Lower values detect more objects but increase false positives. Start at 0.24 and raise to 0.35–0.45 to reduce noise.
- **Prompt specificity**: `"large building"` works better than `"building"` in dense urban scenes.
- **Model variant**: `LangSAM(sam_type="vit_h")` uses the highest-quality SAM backbone at the cost of ~2.4 GB RAM.